# Trend Raster Hotspot Analysis Workflow

Some notes:
- This notebook is meant to be reproducible from within any Pro project on any computer. However, it will not function outside of ArcGIS Pro.
- It is meant to be iterable: it should be possible to use this to process multiple datasets with a variety of parameters. 
- It assumes that this notebook has been placed in the project folder, next to the .aprx file.
- Users are asked to specify some parameters in the User Configuration section of the notebook. Other than that, it should be possible to run the entire workflow without editing. 

## Setup

### Import modules

In [31]:
import arcpy
import os
from arcpy.sa import Con
arcpy.CheckOutExtension("Spatial")
import math

### Project context & environment

In [32]:
# Get the Pro project we're currently using
aprx = arcpy.mp.ArcGISProject("CURRENT")

# Setting the project directory for use in creating relative paths later
# Important: this assumes that this notebook is in the project directory!
# change to project directory and then save cwd to variable "project_dir"
os.chdir(aprx.homeFolder)
project_dir = os.getcwd()

# Get the project's default geodatabase, where we'll be saving some output
gdb_path = aprx.defaultGeodatabase

# set overwrites
arcpy.env.overwriteOutput = True

# Report 
print("Project file:", aprx.filePath)
print("Project directory:", project_dir)
print("Workspace:", gdb_path)

Project file: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\TrendRasterHotspotAnalysis.aprx
Project directory: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis
Workspace: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\TrendRasterHotspotAnalysis.gdb


#### Emergency file path hardcoding

This notebook is meant to automatically detect your project file and build the folder structure around that. However, you can hardcode file paths here if you really need to. 

In [33]:
# your project's absolute path
# aprx = "C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\TrendRasterHotspotAnalysis.aprx"

# Important: this assumes that this notebook is in the project directory!
# os.chdir(aprx.homeFolder)
# project_dir = os.getcwd()
#gdb_path = aprx.defaultGeodatabase
# Report 
#print("Project file:", aprx.filePath)
#print("Project directory:", project_dir)
#print("Workspace:", gdb_path)

### Helper functions

#### Build path function

In [34]:
def build_path(base, rel_path):
    return os.path.abspath(os.path.join(base, rel_path))

# demo
example_rel_path = "example_file.tif"
example_abs_path = build_path(project_dir, example_rel_path)
print(example_abs_path)

C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\example_file.tif


#### Validation functions
Just setting up the functions that will be used later. Validation doesn't actually occur when you run this cell.

In [35]:
# folder validation and creation
def validate_or_create_folder(folder_path, description, create=True):
    """
    Validate that a folder exists.
    If create=True, create it if missing.
    If create=False, raise an error if missing.
    """
    if os.path.isdir(folder_path):
        print(f"{description} folder exists: {folder_path}")
        return

    if create:
        os.makedirs(folder_path, exist_ok=True)
        print(f"[INFO] Created {description} folder: {folder_path}")
    else:
        raise FileNotFoundError(
            f"{description} folder does not exist: {folder_path}"
        )

# checks that files exist
def validate_file(path, description):
    if not arcpy.Exists(path) and not os.path.isfile(path):
        raise FileNotFoundError(f"{description} file not found: {path}")
    print(f"{description} file found: {path}")

# check if vector dataset is in target CRS
def validate_vector_crs(in_features, target_crs, description="Input features"):
    """
    Validate that vector data has a defined CRS and matches the target CRS.
    Fails loudly if not.
    """
    desc = arcpy.Describe(in_features)
    sr = desc.spatialReference

    if sr is None or sr.name == "Unknown":
        raise ValueError(
            f"{description} has undefined coordinate system: {in_features}"
        )

    target_sr = target_crs

    if sr.factoryCode != target_sr.factoryCode:
        raise ValueError(
            f"{description} CRS mismatch.\n"
            f"Expected: {target_sr.name} (EPSG:{target_sr.factoryCode})\n"
            f"Found: {sr.name} (EPSG:{sr.factoryCode})\n"
            f"Fix this by projecting the data BEFORE running the workflow."
        )

    # If we get here, CRS is valid
    print(f"{description} CRS validated ({sr.name})")

## **User Instructions**

#### Expected folder structure

Create subfolders within your project folder with the structure and names shown in the next cell, or edit the default folder variables in the User Configuration section if you'd like to customize it. 

In [36]:
# Project/
    # data/
        # aoi/
        # input_rasters/
    # output/
        # aggregation/
        # hotspots/
    # TrendRasterHotspots.ipynb

#### Expected input files and parameter instructions

This project takes as input:  
- 1 area of interest as a shapefile, which should be place in the aoi subfolder
- A list of rasters exported from google earth engine in any CRS, located in the input_rasters folder
- A list of ratios of pixels:hexagons you want to try
- A list of numbers of nearest neighbors you want to try using for hotspot analysis
- A coordinate reference system
- A naming prefix
- Good thoughts & gratitude 

Output files will be named with the following convention:   
{prefix}_raster   
{prefix}_hex{ratio}_unclipped  
{prefix}_hex{ratio}_clipped  
{prefix}_hex{ratio}_sum  
{prefix}_hex{ratio}_hotspots_{k}n

## **User Configuration**

**Important**: Please review the *User Instructions* cells above for context before editing the following two cells. 

#### **EDIT THIS CELL ONLY**

In [37]:
### INPUT FILES ###

# specify area of interest file, located in aoi subfolder 
aoi_filename = "TC_boundary.shp"

# specify LandTrendr rasters to combine conditionally, located in the "input_rasters" folder
# you can use any number of rasters, but recommend less than 6. 
# to use only 1 raster, make a 1-item list
input_rasters = ["TC_PersistentGreening.tif", "TC_Greening2.tif", "TC_Greening3.tif"]

### PARAMETERS ###

# output file naming convention prefix 
name_prefix = "tc_greening"

# coordinate reference system. Default is UTM Zone 15
target_crs = arcpy.SpatialReference(26915)

# list: ratios of hexagon width to pixel width for aggregation
hex_ratios = [3, 5, 7, 9]

# list: number of nearest neighbors for hotspot analysis
# multiples of 6 are best
neighbors = [3, 6, 12, 15]

### DEFAULT SUBFOLDER RELATIVE PATHS ###
# These follow the expected folder structure. Recommend no edits. 

data_rel_path = "data" # data folder name
output_rel_path = "output" # output folder name
aoi_rel_path = "aoi" # aoi folder name
input_rasters_rel_path = "input_rasters" # input rasters folder name
aggregation_rel_path = "aggregation" # aggregation output subfolder name
hotspots_rel_path = "hotspots" # hotspots output subfolder name

### Resolve & Validate

#### Resolve file paths

In [38]:
# resolve absolute paths from specified relative paths
data_dir = build_path(project_dir, data_rel_path)
output_dir = build_path(project_dir, output_rel_path)

# paths within data folder
input_rasters_dir = build_path(data_dir, input_rasters_rel_path)
aoi_dir = build_path(data_dir, aoi_rel_path)

# paths within output folder
aggregation_dir = build_path(output_dir, aggregation_rel_path)
hotspots_dir = build_path(output_dir, hotspots_rel_path)

# resolve file paths
aoi_path = build_path(aoi_dir, aoi_filename)

raster_paths = [
    build_path(input_rasters_dir, filename)
    for filename in input_rasters
]

#### Validate folders & create if needed

In [39]:
# Validate folders
folders = [
    (data_dir, "Data"),
    (output_dir, "Output"),
    (input_rasters_dir, "Input rasters"),
    (aoi_dir, "AOI"),
    (aggregation_dir, "Aggregation"),
    (hotspots_dir, "Hotspots")
]

for folder, desc in folders:
    validate_or_create_folder(folder, desc, create=True)

Data folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\data
Output folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\output
Input rasters folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\data\input_rasters
AOI folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\data\aoi
Aggregation folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\output\aggregation
Hotspots folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\output\hotspots


#### Validate file existence & coordinate system 

In [40]:
# Validate files
validate_file(aoi_path, "AOI")

for p in raster_paths:
    validate_file(p, "Raster")

# validate CRS for vector data
validate_vector_crs(
    aoi_path,
    target_crs,
    description="AOI"
)

AOI file found: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\data\aoi\TC_boundary.shp
Raster file found: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\data\input_rasters\TC_PersistentGreening.tif
Raster file found: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\data\input_rasters\TC_Greening2.tif
Raster file found: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\data\input_rasters\TC_Greening3.tif
AOI CRS validated (NAD_1983_UTM_Zone_15N)


## Functions

### Step 1: Prepare the binary trend rasters

LandTrendr analyses produce multiple rasters, each with binary values that indicate the presence or absence of a trend. 

This function should allow a user to specify the file path of several rasters, then combine then into a set of values.

The sample data product we're using has 6 classes, 3 of which we're using to indicate "greening present" and the other 3 we're ignoring for now. We're going to combine those 3 and reclassify them as just 0 or 1. 

#### Load and combine the raw trend rasters

In [41]:
def combine_trend_rasters(raster_paths, out_raster):
    
    # loop through list of input rasters 
    rasters = [arcpy.Raster(p) for p in raster_paths]
    # set the reference raster
    ref = rasters[0]

    # align everything to the reference raster 
    arcpy.env.snapRaster = ref
    arcpy.env.cellSize = ref
    arcpy.env.extent = ref

    summed = rasters[0]
    for r in rasters[1:]:
        summed += r

    #Combine a list of rasters conditionally: >0 becomes 1, else 0.
    combined = Con(summed > 0, 1, 0)
    combined.save(out_raster)
    
    return combined

#### Wrapper function: combines, projects & clips the input rasters

In [42]:
def prepare_trend_raster(raster_paths, aoi_path, gdb_path, name_prefix, out_crs):
    #Combines rasters conditionally, then projects and clips to AOI.
    #Intermediate rasters are saved in the default GDB.
    print("Loading rasters...")
    
    # Step 1: Combine
    combined_raster = build_path(gdb_path, f"{name_prefix}_combined")
    print("Combining and reclassifying input rasters...")
    combine_trend_rasters(raster_paths, combined_raster)
    print("Rasters combined.")
    
    # Step 2: Project (call ArcPy directly)
    print("Projecting raster to target CRS...")
    projected_raster = build_path(gdb_path, f"{name_prefix}_projected")
    arcpy.management.ProjectRaster(
        combined_raster,
        projected_raster,
        out_coor_system=out_crs,
        resampling_type="NEAREST"
    )
    print("Raster projected.")
    
    # Step 3: Clip (call ArcPy directly)
    print("Clipping raster to area of interest...")
    binary_raster = build_path(gdb_path, f"{name_prefix}_binary_clipped")
    arcpy.management.Clip(
        projected_raster,
        "#",  # use full extent from AOI
        binary_raster,
        aoi_path,
        "#",
        "ClippingGeometry",
        "MAINTAIN_EXTENT"
    )

    print("Binary trend raster ready for further processing:")
    print(binary_raster)
    
    return binary_raster

In [43]:
# Test call

#print("Running test: prepare_trend_raster")

#binary_raster = prepare_trend_raster(raster_paths=raster_paths,aoi_path=aoi_path, gdb_path=gdb_path,name_prefix=name_prefix,out_crs=target_crs)

### Step 2: Aggregate the binary trend data

#### Calculate hexagon areas for each pixel:hexagon width ratio

In [44]:
def hex_area_from_ratio(binary_raster, ratio):
    # describe raster to get pixel width
    r_desc = arcpy.Describe(binary_raster)
    pixel_width = round(r_desc.children[0].meanCellWidth, 2)

    hex_width = pixel_width * ratio       # vertex-to-vertex width
    hex_side_length = hex_width / 2       # side length
    hex_area = (3 * math.sqrt(3) / 2) * (hex_side_length ** 2)

    info = {
        "ratio": ratio,
        "hex_width": hex_width,
        "hex_side_length": hex_side_length,
        "hex_area": hex_area
    }

    print(f"For hexagons {ratio}x the pixel width ({pixel_width}m):")
    print(f"  Hex width (vertex-to-vertex): {hex_width:.2f} m")
    print(f"  Hex area:                     {hex_area:.2f} sq m")
    print("-" * 60)

    return info

In [45]:
# test call

#for ratio in hex_ratios:
#    hex_area_from_ratio(binary_raster, ratio)

#### Create & clip tessellated hexagons for each ratio

In [46]:
def create_and_clip_hexagons(binary_raster, aoi_path, hex_info_list, gdb_path, name_prefix):
    """
    Generate hexagon tessellations for each hex info dictionary, clipped to AOI.

    Parameters:
      - binary_raster: raster used to determine spatial reference
      - aoi_path: polygon for clipping
      - hex_info_list: list of dictionaries from hex_area_from_ratio()
      - gdb_path: where to save outputs
      - name_prefix: prefix for output feature class names

    Returns:
      dict: {ratio: {"unclipped": fc_path, "clipped": fc_path}}
    """
    hex_outputs = {}

    for info in hex_info_list:
        ratio = info["ratio"]
        hex_area_val = info["hex_area"]

        print(f"Generating hexagons for ratio {ratio}x pixel width...")

        # feature class names/paths
        # I was going to write the unclipped hexagons just to memory, but what if someone tries to do a huge area? 
        fc_unclipped = build_path(gdb_path, f"{name_prefix}_hex{ratio}_unclipped")
        fc_clipped = build_path(gdb_path, f"{name_prefix}_hex{ratio}_clipped")

        # generate tessellation
        arcpy.management.GenerateTessellation(
            Output_Feature_Class=fc_unclipped,
            Extent=aoi_path,
            Shape_Type="HEXAGON",
            Size=hex_area_val,
            Spatial_Reference=target_crs
        )

        # clip to AOI
        arcpy.analysis.Clip(
            in_features=fc_unclipped,
            clip_features=aoi_path,
            out_feature_class=fc_clipped
        )
        print(f"Clipped {ratio}x hexagons saved.")

        # save paths to dict
        hex_outputs[ratio] = fc_clipped

    return hex_outputs

In [47]:
# test call

#hex_info_list = [hex_area_from_ratio(binary_raster, r) for r in hex_ratios]

#hex_meshes = create_and_clip_hexagons(binary_raster, aoi_path, hex_info_list, gdb_path, name_prefix)


#### Sum binary raster values and join to hexagon 

In [48]:
def sum_raster_to_hexagons(hex_dict, binary_raster, aggregation_dir, name_prefix):
    """
    Summarize binary raster values within each hexagon and join to hex polygons.

    Parameters:
      - hex_dict: {ratio: clipped_hex_fc_path}
      - binary_raster: raster to aggregate
      - aggregation_dir: folder to save outputs
      - name_prefix: prefix for output filenames

    Returns:
      dict: {ratio: aggregated_hex_fc_path}
    """
    aggregated_hexes = {}

    for ratio, hex_fc in hex_dict.items():
        
        print(f"Summing raster values within {ratio}x hexagons...")
        
        # output table path
        zonal_table = build_path(aggregation_dir, f"{name_prefix}_hex{ratio}_zonal_table.dbf")
        # aggregated_fc path
        aggregated_fc = build_path(aggregation_dir, f"{name_prefix}_hex{ratio}_sum")
        # copy features to new path
        arcpy.management.CopyFeatures(hex_fc, aggregated_fc)
        # zone fields are kind of unstable 
        # I was using "FID" but i got errors, so now it's based on Describe. 
        hex_zone_field = arcpy.Describe(aggregated_fc).OIDFieldName
        
        # ZonalStatisticsAsTable
        arcpy.sa.ZonalStatisticsAsTable(
            in_zone_data=aggregated_fc,
            zone_field=hex_zone_field,
            in_value_raster=binary_raster,
            out_table=zonal_table,
            statistics_type="SUM",
            ignore_nodata="DATA"
        )

        # Detect the join field in the table automatically
        # this has been so unstable! I'm pulling out the big guns! 
        possible_join_fields = [f.name for f in arcpy.ListFields(zonal_table) 
                        if f.name.startswith(hex_zone_field)]
        if not possible_join_fields:
            raise ValueError(f"No matching OID field found in zonal table for {hex_zone_field}")
        table_zone_field = possible_join_fields[0]

        # join SUM back to hex polygons
        print("Joining sums to hexagons...")
        
        #more field logic
        #sum_zone_field = arcpy.Describe(aggregated_fc).OIDFieldName
        # now we can join 
        arcpy.management.JoinField(
            in_data=aggregated_fc,
            in_field=hex_zone_field,
            join_table=zonal_table,
            join_field=table_zone_field,  # field in table
            fields=["SUM"]
        )
        print(f"Aggregated hexagons saved.")

        aggregated_hexes[ratio] = aggregated_fc
    print("Aggregation complete.") 
    return aggregated_hexes

In [49]:
#test call
#aggregated_hexes = sum_raster_to_hexagons(hex_meshes, binary_raster, aggregation_dir, name_prefix)

#### Wrapper function: Hexagon Aggregation

In [50]:
def hexagon_aggregation(binary_raster, aoi_path, hex_ratios, gdb_path, name_prefix):
    # calculate hexagon areas for specified ratios, saved to a list
    hex_info_list = [hex_area_from_ratio(binary_raster, r) for r in hex_ratios]

    # create a dictionary of hexagon meshes clipped to aoi based on list of hexagon areas
    hex_meshes = create_and_clip_hexagons(binary_raster, aoi_path, hex_info_list, gdb_path, name_prefix)

    # sum rasters within hexagons and join to hexagons
    aggregated_hexes = sum_raster_to_hexagons(hex_meshes, binary_raster, aggregation_dir, name_prefix)

    return aggregated_hexes

In [51]:
#test call
# aggregated_hexes = hexagon_aggregation(binary_raster, aoi_path, hex_ratios, gdb_path, name_prefix)

### Step 3: Hotspot analysis

#### Hotspots for a single aggregation

In [52]:
def hotspot_single_analysis(in_fc, out_fc, number_of_neighbors):
    arcpy.stats.HotSpots(
                Input_Feature_Class=in_fc,
                Input_Field="SUM",
                Output_Feature_Class=out_fc,
                Conceptualization_of_Spatial_Relationships="K_NEAREST_NEIGHBORS",
                Distance_Method="EUCLIDEAN_DISTANCE",
                Standardization="ROW",
                Apply_False_Discovery_Rate__FDR__Correction="APPLY_FDR",
                number_of_neighbors=number_of_neighbors
            )
    return out_fc

#### Wrapper function: hotspots for all aggregations. 

In [53]:
def hotspot_analysis(
    aggregated_hexes,
    neighbors,
    hotspots_dir,
    name_prefix,
    value_field="SUM"
):
    """
    Run hotspot analysis on aggregated hexagon layers.

    Parameters:
      - aggregated_hexes: {ratio: hex_fc_with_SUM}
      - neighbors_list: list of neighbor counts
      - fdr: True / False
      - hotspots_dir: output folder
      - name_prefix: naming prefix

    Returns:
      dict: {(ratio, neighbors): hotspot_fc_path}
    """
        
    hotspot_outputs = {}

    for ratio, aggregated_fc in aggregated_hexes.items():
        for k in neighbors:

            print(f"Running hotspot analysis: {ratio}x hexagons, {k} neighbors...")

            out_fc = build_path(
                hotspots_dir,
                f"{name_prefix}_hex{ratio}_hotspots_{k}n"
            )

            arcpy.stats.HotSpots(
                Input_Feature_Class=aggregated_fc,
                Input_Field="SUM",
                Output_Feature_Class=out_fc,
                Conceptualization_of_Spatial_Relationships="K_NEAREST_NEIGHBORS",
                Distance_Method="EUCLIDEAN_DISTANCE",
                Standardization="ROW",
                Apply_False_Discovery_Rate__FDR__Correction="APPLY_FDR",
                number_of_neighbors=k
            )

            hotspot_outputs[(ratio, k)] = out_fc
            print("Hotspot analysis complete.")

    return hotspot_outputs

In [54]:
def main():
    # -----------------------------
    # 1. Prepare binary trend raster
    # -----------------------------
    binary_raster = prepare_trend_raster(
        raster_paths=raster_paths,
        aoi_path=aoi_path,
        gdb_path=gdb_path,
        name_prefix=name_prefix,
        out_crs=target_crs
    )

    # -----------------------------
    # 2. Hexagon aggregation
    # -----------------------------
    aggregated_hexes = hexagon_aggregation(
        binary_raster=binary_raster,
        aoi_path=aoi_path,
        hex_ratios=hex_ratios,
        gdb_path=gdb_path,
        name_prefix=name_prefix
    )
    print("Aggregated hexagons in /output/aggregation")

    # -----------------------------
    # 3. Hotspot analysis
    # -----------------------------
    hotspot_outputs = hotspot_analysis(
        aggregated_hexes=aggregated_hexes,
        neighbors=neighbors,
        hotspots_dir=hotspots_dir,
        name_prefix=name_prefix,
        value_field="SUM"
    )
    print("Hotspot layers in /output/hotspots")

    return binary_raster, aggregated_hexes, hotspot_outputs

In [55]:
if __name__ == "__main__":
    binary_raster, aggregated_hexes, hotspot_outputs = main()

Loading rasters...
Combining and reclassifying input rasters...
Rasters combined.
Projecting raster to target CRS...
Raster projected.
Clipping raster to area of interest...
Binary trend raster ready for further processing:
C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\TrendRasterHotspotAnalysis.gdb\tc_greening_binary_clipped
For hexagons 3x the pixel width (25.38m):
  Hex width (vertex-to-vertex): 76.14 m
  Hex area:                     3765.46 sq m
------------------------------------------------------------
For hexagons 5x the pixel width (25.38m):
  Hex width (vertex-to-vertex): 126.90 m
  Hex area:                     10459.60 sq m
------------------------------------------------------------
For hexagons 7x the pixel width (25.38m):
  Hex width (vertex-to-vertex): 177.66 m
  Hex area:                     20500.82 sq m
------------------------------------------------------------
For hexagons 9x the pixel width (25.38m):
  Hex width (vertex-to-vertex): 228.42 m